# Cien millones no son entropía

## PoC reproducible en macOS y Apple Silicon

Este notebook demuestra, con **doce identidades completamente sintéticas**, que
el ancho de una salida SHA-256 no aumenta la entropía del identificador de
entrada.

El recorrido:

1. Comprueba el entorno y compila el enumerador.
2. Calcula el dominio real de DNI y NIE.
3. Genera siete tablas sintéticas.
4. Recorre los 100.000.000 de DNI posibles.
5. Reconstruye doce perfiles mediante tres caminos independientes.
6. Genera figuras y un manifiesto para documentar el artículo.

> **Uso ético:** no introduzcas hashes procedentes de filtraciones ni
> identificadores asociados a personas reales.


### Cómo ejecutar

Abre este notebook desde la raíz del proyecto:

```bash
jupyter lab notebooks/Fulcrum_DNI_PoC_Mac.ipynb
```

Ejecuta las celdas en orden. La prueba principal está activada. Las mediciones
de un solo hilo y sal global conocida también lo están porque respaldan dos
afirmaciones del artículo; puedes desactivarlas en la celda siguiente.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import sqlite3
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pandas as pd

try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class HTML(str):
        pass
    class Markdown(str):
        pass
    def display(value):
        print(value)


def locate_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current / "fulcrum-dni-research"]
    for candidate in candidates:
        if (candidate / "src" / "dni_sha256_enum.c").is_file():
            return candidate
    raise RuntimeError(
        "No encuentro la raíz de fulcrum-dni-research. "
        "Abre Jupyter desde la carpeta del proyecto."
    )


ROOT = locate_root()
os.chdir(ROOT)

VALIDATE_ONLY = os.environ.get("POC_NOTEBOOK_VALIDATE_ONLY") == "1"
RUN_SINGLE_THREAD = False
RUN_KNOWN_SALT_BENCHMARK = False
THREADS_ALL = max(1, os.cpu_count() or 1)

FIGURES = (
    Path("/tmp/fulcrum-notebook-figures")
    if VALIDATE_ONLY
    else ROOT / "results" / "figures_mac"
)
FIGURES.mkdir(parents=True, exist_ok=True)

NAVY = "#0b132b"
BLUE = "#2563eb"
CYAN = "#22d3ee"
GREEN = "#22c55e"
AMBER = "#f59e0b"
RED = "#ef4444"
LIGHT = "#e5eefb"
MUTED = "#9fb3c8"

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.facecolor": NAVY,
        "figure.facecolor": NAVY,
        "text.color": LIGHT,
        "axes.labelcolor": LIGHT,
        "xtick.color": LIGHT,
        "ytick.color": LIGHT,
    }
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def first_line(command: list[str]) -> str:
    try:
        completed = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )
        return (completed.stdout or completed.stderr).splitlines()[0]
    except Exception:
        return "No disponible"


def save_figure(fig, filename: str) -> Path:
    path = FIGURES / filename
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
    return path


def show_table(frame: pd.DataFrame) -> None:
    try:
        display(frame.style.hide(axis="index"))
    except ImportError:
        display(frame.to_string(index=False))


print("Proyecto localizado:", ROOT.name)
print("Python:", sys.version.split()[0])
print("Arquitectura:", platform.machine())
print("Hilos lógicos detectados:", THREADS_ALL)


## 1. Comprobación del entorno

En macOS se necesitan las herramientas de Xcode, Homebrew, OpenSSL 3 y
`pkg-config`. Esta celda no instala nada automáticamente: comprueba el entorno y
muestra la orden exacta si falta una dependencia.


In [ ]:
system = platform.system()
machine = platform.machine()
brew_path = shutil.which("brew")

if system == "Darwin" and brew_path:
    try:
        openssl_prefix = subprocess.run(
            [brew_path, "--prefix", "openssl@3"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
        pkgconfig_dir = str(Path(openssl_prefix) / "lib" / "pkgconfig")
        current_pkgconfig = os.environ.get("PKG_CONFIG_PATH", "")
        os.environ["PKG_CONFIG_PATH"] = (
            pkgconfig_dir
            if not current_pkgconfig
            else pkgconfig_dir + os.pathsep + current_pkgconfig
        )
    except subprocess.CalledProcessError:
        pass

tool_rows = [
    {
        "Componente": "Python >= 3.11",
        "Disponible": sys.version_info >= (3, 11),
        "Requerido": True,
    },
    {
        "Componente": "make",
        "Disponible": shutil.which("make") is not None,
        "Requerido": True,
    },
    {
        "Componente": "clang/cc",
        "Disponible": bool(shutil.which("clang") or shutil.which("cc")),
        "Requerido": True,
    },
    {
        "Componente": "pkg-config",
        "Disponible": shutil.which("pkg-config") is not None,
        "Requerido": system == "Darwin",
    },
]

openssl_version = "No disponible"
if shutil.which("pkg-config"):
    openssl_version = first_line(["pkg-config", "--modversion", "openssl"])
tool_rows.append(
    {
        "Componente": f"OpenSSL ({openssl_version})",
        "Disponible": openssl_version != "No disponible",
        "Requerido": system == "Darwin",
    }
)

environment_table = pd.DataFrame(tool_rows)
environment_table["Estado"] = environment_table.apply(
    lambda row: (
        "OK"
        if row["Disponible"]
        else ("FALTA" if row["Requerido"] else "OPCIONAL")
    ),
    axis=1,
)
environment_table = environment_table[["Componente", "Estado"]]
show_table(environment_table)

READY = not (environment_table["Estado"] == "FALTA").any()
if not READY:
    print("\nInstala los componentes que faltan:")
    if system == "Darwin":
        print("  xcode-select --install")
        print("  brew install openssl@3 pkg-config")
        print("Después reinicia el kernel y vuelve a ejecutar esta celda.")
else:
    print("\nEntorno preparado para compilar.")


## 2. Compilación y pruebas rápidas

Se compila el enumerador C con OpenSSL y hilos POSIX. Después se ejecutan las
pruebas de formato, letra de control, ofuscación y reconstrucción sobre un
subconjunto pequeño.


In [ ]:
if not READY:
    raise RuntimeError("Faltan dependencias. Revisa la celda anterior.")

build = subprocess.run(
    ["make"],
    cwd=ROOT,
    env=os.environ.copy(),
    check=True,
    capture_output=True,
    text=True,
)
print(build.stdout.strip())

tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    env=os.environ.copy(),
    check=True,
    capture_output=True,
    text=True,
)
test_output = tests.stdout + tests.stderr
summary_lines = [
    line for line in test_output.splitlines()
    if line.startswith("Ran ") or line.strip() == "OK"
]
print("\nPruebas:", " | ".join(summary_lines))

cracker = ROOT / "build" / "dni_sha256_enum"
assert cracker.is_file(), "No se ha generado el enumerador"


## 3. Figura 1 — El dominio real

La letra de control del DNI es determinista. La salida de SHA-256 ocupa 256
bits, pero el conjunto máximo de entradas sigue siendo de 100.000.000, unos
26,58 bits.

Esta figura acompaña la sección **«La pregunta incómoda: 26,6 bits»**.


In [ ]:
dni_domain = 100_000_000
nie_domain = 30_000_000
dni_bits = math.log2(dni_domain)
nie_bits = math.log2(nie_domain)

fig, ax = plt.subplots(figsize=(13.5, 5.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

cards = [
    (0.035, "DNI/NIF", "100.000.000", f"{dni_bits:.2f} bits", BLUE),
    (0.355, "NIE", "30.000.000", f"{nie_bits:.2f} bits", CYAN),
    (0.675, "Salida SHA-256", "256 bits", "No añade entropía", AMBER),
]
for x, title, value, detail, color in cards:
    patch = FancyBboxPatch(
        (x, 0.2),
        0.29,
        0.62,
        boxstyle="round,pad=0.012,rounding_size=0.03",
        linewidth=2,
        edgecolor=color,
        facecolor="#111c36",
    )
    ax.add_patch(patch)
    ax.text(x + 0.145, 0.69, title, ha="center", fontsize=15, color=MUTED)
    ax.text(x + 0.145, 0.49, value, ha="center", fontsize=24, weight="bold", color=color)
    ax.text(x + 0.145, 0.32, detail, ha="center", fontsize=13, color=LIGHT)

ax.text(
    0.5,
    0.06,
    "El ancho del hash no es el tamaño del secreto",
    ha="center",
    fontsize=17,
    weight="bold",
    color=LIGHT,
)
fig.suptitle(
    "Cien millones no son entropía",
    fontsize=25,
    weight="bold",
    color=LIGHT,
    y=0.98,
)
save_figure(fig, "01_dominio_vs_hash.png")
plt.show()


## 4. Generación del laboratorio sintético

Se crean doce identidades ficticias distribuidas entre siete tablas. Los nombres
usan el prefijo `PERSONA_SINTETICA`, los correos terminan en `.invalid` y los
IBAN y pólizas son sustitutos manifiestamente no válidos.


In [ ]:
if VALIDATE_ONLY:
    database = ROOT / "lab" / "synthetic_identity.db"
    generation = {"synthetic": True, "database": database.name}
else:
    database = ROOT / "lab" / "synthetic_identity_mac.db"
    generated = subprocess.run(
        [
            sys.executable,
            "src/generate_lab.py",
            "--output",
            str(database.relative_to(ROOT)),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    generation = json.loads(generated.stdout)

connection = sqlite3.connect(database)
tables = [
    "staging_identity",
    "customer_core",
    "contact_details",
    "financial_profile",
    "case_notes",
    "auth_users",
    "defended_identifiers",
]
counts = {
    table: connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    for table in tables
}
connection.close()

assert generation["synthetic"] is True
assert all(value == 12 for value in counts.values())
show_table(
    pd.DataFrame(
        [{"Tabla": table, "Registros sintéticos": count} for table, count in counts.items()]
    )
)
print("Base validada: siete tablas × doce registros sintéticos.")


### Figura 2 — La clave de enlace

El diagrama muestra por qué proteger una columna aislada no limita necesariamente
el alcance: `link_key` permite saltar entre las tablas y staging conserva una
copia en claro.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

nodes = {
    "staging_identity": (0.05, 0.69, "NIF en claro", RED),
    "customer_core": (0.37, 0.69, "SHA-256(NIF)", AMBER),
    "auth_users": (0.69, 0.69, "scrypt", GREEN),
    "contact_details": (0.05, 0.16, "Email · teléfono · póliza", BLUE),
    "financial_profile": (0.37, 0.16, "Perfil financiero", CYAN),
    "case_notes": (0.69, 0.16, "Ofuscación reversible", RED),
}

for name, (x, y, detail, color) in nodes.items():
    patch = FancyBboxPatch(
        (x, y),
        0.26,
        0.15,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=2,
        edgecolor=color,
        facecolor="#111c36",
    )
    ax.add_patch(patch)
    ax.text(x + 0.13, y + 0.098, name, ha="center", fontsize=13, weight="bold")
    ax.text(x + 0.13, y + 0.048, detail, ha="center", fontsize=10.5, color=MUTED)

center = FancyBboxPatch(
    (0.36, 0.43),
    0.28,
    0.13,
    boxstyle="round,pad=0.012,rounding_size=0.03",
    linewidth=2.5,
    edgecolor=LIGHT,
    facecolor=BLUE,
)
ax.add_patch(center)
ax.text(0.5, 0.495, "link_key", ha="center", va="center", fontsize=18, weight="bold")

for name, (x, y, _, _) in nodes.items():
    target = (x + 0.13, y + (0 if y > 0.5 else 0.15))
    source = (0.5, 0.56 if y > 0.5 else 0.43)
    ax.annotate(
        "",
        xy=target,
        xytext=source,
        arrowprops={"arrowstyle": "->", "color": MUTED, "lw": 1.7},
    )

fig.suptitle(
    "Una clave estable reconstruye el perfil completo",
    fontsize=24,
    weight="bold",
    y=0.96,
)
ax.text(
    0.5,
    0.06,
    "El alcance lo determina la copia y la relación más débiles",
    ha="center",
    fontsize=15,
    color=LIGHT,
    weight="bold",
)
save_figure(fig, "02_modelo_datos.png")
plt.show()


## 5. Ejecución principal — 100.000.000 de candidatos

Esta es la medición central. El enumerador utiliza todos los hilos lógicos
detectados, calcula SHA-256 para cada DNI sintácticamente posible y compara cada
digest con los doce objetivos durante la misma pasada.

La celda no imprime rutas locales ni nombres de usuario. El resultado completo
se guarda en `results/reconstruction_mac_multithread.json`.


In [ ]:
if VALIDATE_ONLY:
    report_path = ROOT / "results" / "reconstruction.json"
else:
    report_path = ROOT / "results" / "reconstruction_mac_multithread.json"
    command = [
        sys.executable,
        "src/run_reconstruction.py",
        "--database",
        str(database.relative_to(ROOT)),
        "--cracker",
        "build/dni_sha256_enum",
        "--output",
        str(report_path.relative_to(ROOT)),
        "--limit",
        "100000000",
        "--threads",
        str(THREADS_ALL),
    ]
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=os.environ.copy(),
        check=True,
        capture_output=True,
        text=True,
    )
    wall_seconds = time.perf_counter() - started
    print("Ejecución completa finalizada.")
    print("Tiempo total del wrapper:", f"{wall_seconds:.2f} s")

report = json.loads(report_path.read_text(encoding="utf-8"))
assert report["target_hashes"] == 12
assert report["recovered_hashes"] == 12
assert report["consistency_failures"] == []
assert len(report["identity_packages"]) == 12

print("Objetivos:", report["target_hashes"])
print("Recuperados:", report["recovered_hashes"])
print("Fallos de consistencia:", len(report["consistency_failures"]))


### Figura 3 — Resultado experimental

Esta es la captura principal del artículo. Contiene el espacio recorrido,
objetivos recuperados, tiempo, throughput y número de hilos, y deja visible que
los datos son sintéticos.


In [ ]:
cracking = report["cracking"]
metrics = [
    ("Candidatos", f'{cracking["candidate_limit"]:,}'.replace(",", "."), BLUE),
    ("Recuperados", f'{report["recovered_hashes"]}/{report["target_hashes"]}', GREEN),
    ("Tiempo", f'{cracking["elapsed_seconds"]:.2f} s', AMBER),
    ("Throughput", f'{cracking["hashes_per_second"]/1_000_000:.2f} M/s', CYAN),
]

fig, ax = plt.subplots(figsize=(15, 6.3))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

for index, (title, value, color) in enumerate(metrics):
    x = 0.025 + index * 0.245
    patch = FancyBboxPatch(
        (x, 0.29),
        0.22,
        0.42,
        boxstyle="round,pad=0.012,rounding_size=0.025",
        linewidth=2,
        edgecolor=color,
        facecolor="#111c36",
    )
    ax.add_patch(patch)
    ax.text(x + 0.11, 0.61, title, ha="center", fontsize=13, color=MUTED)
    ax.text(x + 0.11, 0.45, value, ha="center", fontsize=21, weight="bold", color=color)

ax.text(
    0.5,
    0.89,
    "Enumeración completa de SHA-256(DNI)",
    ha="center",
    fontsize=25,
    weight="bold",
)
ax.text(
    0.5,
    0.17,
    f'Hilos: {cracking["threads"]}  ·  Formato: 8 dígitos + letra módulo 23',
    ha="center",
    fontsize=13,
    color=MUTED,
)
ax.text(
    0.5,
    0.07,
    "LABORATORIO CON DATOS ÍNTEGRAMENTE SINTÉTICOS",
    ha="center",
    fontsize=13,
    color=GREEN,
    weight="bold",
)
save_figure(fig, "03_resultado_reconstruccion.png")
plt.show()


## 6. Figura 4 — Tres caminos independientes

La PoC separa la enumeración del hash, la vinculación mediante staging y la
reversión de una ofuscación débil. La coexistencia de los tres caminos evita
atribuir todo el compromiso a una única columna.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

paths = [
    (
        0.035,
        "1 · Enumeración",
        "customer_core\nSHA-256(DNI)",
        "100 M candidatos",
        "DNI recuperado",
        AMBER,
    ),
    (
        0.355,
        "2 · Vinculación",
        "staging_identity\nNIF en claro",
        "link_key",
        "Perfil completo",
        BLUE,
    ),
    (
        0.675,
        "3 · Ofuscación",
        "case_notes\nsustitución + hex",
        "Transformación inversa",
        "DNI recuperado",
        RED,
    ),
]

for x, title, source, action, result, color in paths:
    patch = FancyBboxPatch(
        (x, 0.19),
        0.29,
        0.61,
        boxstyle="round,pad=0.012,rounding_size=0.025",
        linewidth=2,
        edgecolor=color,
        facecolor="#111c36",
    )
    ax.add_patch(patch)
    ax.text(x + 0.145, 0.70, title, ha="center", fontsize=16, weight="bold", color=color)
    ax.text(x + 0.145, 0.55, source, ha="center", fontsize=12, color=LIGHT)
    ax.annotate(
        "",
        xy=(x + 0.145, 0.40),
        xytext=(x + 0.145, 0.49),
        arrowprops={"arrowstyle": "->", "color": MUTED, "lw": 2},
    )
    ax.text(x + 0.145, 0.355, action, ha="center", fontsize=11, color=MUTED)
    ax.text(x + 0.145, 0.255, result, ha="center", fontsize=13, weight="bold")

fig.suptitle(
    "No hay una debilidad: hay tres caminos",
    fontsize=25,
    weight="bold",
    y=0.96,
)
ax.text(
    0.5,
    0.07,
    "La seguridad efectiva queda limitada por la copia y la relación más débiles",
    ha="center",
    fontsize=14,
    color=LIGHT,
)
save_figure(fig, "04_tres_caminos.png")
plt.show()


### Evidencia de la reconstrucción multitabla

Se muestran seis de los doce perfiles reconstruidos. Son registros ficticios y
la figura mantiene una marca visible para que nunca pueda confundirse con una
filtración real.


In [ ]:
profiles = pd.DataFrame(report["identity_packages"])
profile_columns = [
    "synthetic_name",
    "recovered_nif",
    "email",
    "policy_number",
    "income_band",
    "risk_score",
]
profile_labels = {
    "synthetic_name": "Identidad",
    "recovered_nif": "NIF recuperado",
    "email": "Email",
    "policy_number": "Póliza",
    "income_band": "Banda",
    "risk_score": "Score",
}
profile_view = profiles[profile_columns].rename(columns=profile_labels)

if not VALIDATE_ONLY:
    profiles.to_csv(
        ROOT / "results" / "reconstructed_profiles_mac.csv",
        index=False,
    )

fig, ax = plt.subplots(figsize=(16, 6.3))
ax.axis("off")
table = ax.table(
    cellText=profile_view.head(6).values,
    colLabels=profile_view.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
    colWidths=[0.18, 0.14, 0.24, 0.19, 0.10, 0.08],
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.75)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#355070")
    if row == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#111c36" if row % 2 else "#172442")
        cell.set_text_props(color=LIGHT)

fig.suptitle(
    "Seis tablas enlazadas · perfiles reconstruidos",
    fontsize=23,
    weight="bold",
    y=0.95,
)
ax.text(
    0.5,
    0.06,
    "DATOS SINTÉTICOS · NO CORRESPONDEN A PERSONAS REALES",
    transform=ax.transAxes,
    ha="center",
    fontsize=13,
    color=GREEN,
    weight="bold",
)
save_figure(fig, "05_perfiles_sinteticos.png")
plt.show()


## 7. Medición opcional con un solo hilo

Esta ejecución permite afirmar exactamente que el resultado se reproduce con
`--threads 1`. No compares su tiempo como si fuera escalado lineal frente al
multihilo: macOS, OpenSSL, temperatura y arquitectura condicionan la medida.


In [ ]:
single_report = None
if VALIDATE_ONLY:
    print("Medición de un hilo omitida durante la validación del notebook.")
elif RUN_SINGLE_THREAD:
    single_path = ROOT / "results" / "reconstruction_mac_1thread.json"
    command = [
        sys.executable,
        "src/run_reconstruction.py",
        "--database",
        str(database.relative_to(ROOT)),
        "--cracker",
        "build/dni_sha256_enum",
        "--output",
        str(single_path.relative_to(ROOT)),
        "--limit",
        "100000000",
        "--threads",
        "1",
    ]
    subprocess.run(
        command,
        cwd=ROOT,
        env=os.environ.copy(),
        check=True,
        capture_output=True,
        text=True,
    )
    single_report = json.loads(single_path.read_text(encoding="utf-8"))
    assert single_report["cracking"]["threads"] == 1
    assert single_report["recovered_hashes"] == 12

    comparison = pd.DataFrame(
        [
            {
                "Ejecución": "Todos los hilos",
                "Hilos": report["cracking"]["threads"],
                "Tiempo (s)": report["cracking"]["elapsed_seconds"],
                "Millones/s": report["cracking"]["hashes_per_second"] / 1_000_000,
                "Recuperados": report["recovered_hashes"],
            },
            {
                "Ejecución": "Un hilo",
                "Hilos": single_report["cracking"]["threads"],
                "Tiempo (s)": single_report["cracking"]["elapsed_seconds"],
                "Millones/s": single_report["cracking"]["hashes_per_second"] / 1_000_000,
                "Recuperados": single_report["recovered_hashes"],
            },
        ]
    )
    show_table(comparison.round(3))

    fig, ax = plt.subplots(figsize=(10, 5.8))
    bars = ax.bar(
        comparison["Ejecución"],
        comparison["Tiempo (s)"],
        color=[BLUE, CYAN],
        width=0.55,
    )
    ax.set_ylabel("Segundos")
    ax.set_title("Mismo dominio y mismos 12 objetivos", fontsize=20, weight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    for bar, value in zip(bars, comparison["Tiempo (s)"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f} s",
            ha="center",
            va="bottom",
            fontsize=12,
            weight="bold",
        )
    ax.text(
        0.5,
        -0.16,
        "DATOS SINTÉTICOS · resultado dependiente del hardware",
        transform=ax.transAxes,
        ha="center",
        color=MUTED,
    )
    save_figure(fig, "08_multihilo_vs_1hilo.png")
    plt.show()
else:
    print("RUN_SINGLE_THREAD=False: medición omitida.")


## 8. Sal global conocida

El benchmark calcula un objetivo sin sal y otro con una sal global conocida.
La sal evita reutilizar una tabla creada para otra transformación, pero el
atacante que conoce la sal puede volver a recorrer exactamente el mismo dominio.

Esta comparación realiza dos pasadas completas adicionales.


In [ ]:
benchmark = None
if VALIDATE_ONLY:
    benchmark_path = ROOT / "results" / "benchmark.json"
    benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
elif RUN_KNOWN_SALT_BENCHMARK:
    benchmark_path = ROOT / "results" / "benchmark_mac.json"
    command = [
        sys.executable,
        "src/benchmark.py",
        "--cracker",
        "build/dni_sha256_enum",
        "--output",
        str(benchmark_path.relative_to(ROOT)),
        "--threads",
        str(THREADS_ALL),
    ]
    subprocess.run(
        command,
        cwd=ROOT,
        env=os.environ.copy(),
        check=True,
        capture_output=True,
        text=True,
    )
    benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
else:
    print("RUN_KNOWN_SALT_BENCHMARK=False: benchmark omitido.")

if benchmark is not None:
    unsalted = benchmark["full_run"]
    salted = benchmark["known_global_salt_full_run"]
    assert unsalted["candidate_limit"] == 100_000_000
    assert salted["candidate_limit"] == 100_000_000
    assert unsalted["found_count"] == salted["found_count"] == 1

    salt_comparison = pd.DataFrame(
        [
            {
                "Transformación": "SHA-256(DNI)",
                "Candidatos": unsalted["candidate_limit"],
                "Tiempo (s)": unsalted["elapsed_seconds"],
                "Recuperado": unsalted["found_count"],
            },
            {
                "Transformación": "SHA-256(sal conocida || DNI)",
                "Candidatos": salted["candidate_limit"],
                "Tiempo (s)": salted["elapsed_seconds"],
                "Recuperado": salted["found_count"],
            },
        ]
    )
    show_table(salt_comparison.round(3))

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.bar(
        ["Sin sal", "Sal global conocida"],
        salt_comparison["Tiempo (s)"],
        color=[BLUE, AMBER],
        width=0.55,
    )
    ax.set_ylabel("Segundos")
    ax.set_title(
        "La sal global conocida no elimina la enumeración",
        fontsize=20,
        weight="bold",
    )
    ax.spines[["top", "right"]].set_visible(False)
    for bar, value in zip(bars, salt_comparison["Tiempo (s)"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f} s",
            ha="center",
            va="bottom",
            fontsize=12,
            weight="bold",
        )
    ax.text(
        0.5,
        -0.16,
        "Ambas pruebas recorren 100.000.000 de candidatos · 1/1 recuperado",
        transform=ax.transAxes,
        ha="center",
        color=MUTED,
    )
    save_figure(fig, "06_salt_global_conocida.png")
    plt.show()


## 9. Figura 7 — Qué cambia cada control

La tabla separa resultados medidos de conclusiones de diseño. HMAC no se
«benchmarkea como resistente» mediante ausencia de coincidencias: su propiedad
es que, sin la clave separada, el atacante no puede calcular los pseudónimos
candidatos.


In [ ]:
controls = pd.DataFrame(
    [
        ["SHA-256(DNI)", "Una pasada sirve para todos los objetivos", "Medido"],
        ["Sal global conocida", "Mismo dominio; evita reutilizar otra tabla", "Medido"],
        ["Sal pública por fila", "Evita amortización masiva; no ataque dirigido", "Análisis"],
        ["HMAC + clave separada", "Sin la clave no se validan candidatos", "Diseño"],
        ["Token aleatorio", "No existe relación matemática que enumerar", "Diseño"],
    ],
    columns=["Control", "Efecto ante extracción", "Base"],
)
show_table(controls)

fig, ax = plt.subplots(figsize=(15, 6.5))
ax.axis("off")
table = ax.table(
    cellText=controls.values,
    colLabels=controls.columns,
    cellLoc="left",
    colLoc="left",
    loc="center",
    colWidths=[0.24, 0.57, 0.12],
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.85)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#355070")
    if row == 0:
        cell.set_facecolor(BLUE)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#111c36" if row % 2 else "#172442")
        cell.set_text_props(color=LIGHT)
        if col == 2:
            cell.set_text_props(color=GREEN if row <= 2 else CYAN, weight="bold")

fig.suptitle(
    "La sal no es lo mismo que el secreto",
    fontsize=24,
    weight="bold",
    y=0.95,
)
ax.text(
    0.5,
    0.06,
    "La eficacia depende también de separación, contexto y custodia de claves",
    transform=ax.transAxes,
    ha="center",
    fontsize=13,
    color=MUTED,
)
save_figure(fig, "07_matriz_controles.png")
plt.show()


## 10. Entorno y manifiesto de evidencia

La última celda registra únicamente información técnica: sistema, arquitectura,
CPU, Python, compilador, OpenSSL, hilos y hashes SHA-256 de código, base
sintética y resultados. No registra nombre de usuario ni hostname.


In [ ]:
cpu_brand = platform.processor() or machine
if system == "Darwin":
    detected_brand = first_line(["sysctl", "-n", "machdep.cpu.brand_string"])
    if detected_brand != "No disponible":
        cpu_brand = detected_brand

environment = {
    "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "system": system,
    "release": platform.release(),
    "machine": machine,
    "cpu": cpu_brand,
    "logical_cpus": os.cpu_count(),
    "python": sys.version.split()[0],
    "compiler": first_line(["clang", "--version"]) if shutil.which("clang") else first_line(["cc", "--version"]),
    "openssl_pkg_config": openssl_version,
    "candidate_format": "eight_digits_plus_deterministic_mod23_letter",
    "candidate_space": 100_000_000,
    "scope": "entirely_synthetic_lab_data",
}

environment_path = (
    Path("/tmp/fulcrum-notebook-environment.json")
    if VALIDATE_ONLY
    else ROOT / "results" / "mac_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

evidence_files = [
    ROOT / "src" / "dni_sha256_enum.c",
    ROOT / "src" / "generate_lab.py",
    ROOT / "src" / "run_reconstruction.py",
    database,
    report_path,
    *sorted(FIGURES.glob("*.png")),
]
if not VALIDATE_ONLY and single_report is not None:
    evidence_files.append(ROOT / "results" / "reconstruction_mac_1thread.json")
if not VALIDATE_ONLY and benchmark is not None:
    evidence_files.append(ROOT / "results" / "benchmark_mac.json")

manifest_lines = [
    f"{sha256_file(path)}  {path.name}"
    for path in evidence_files
    if path.is_file()
]
manifest_path = (
    Path("/tmp/fulcrum-notebook-manifest.sha256")
    if VALIDATE_ONLY
    else ROOT / "results" / "mac_run_manifest.sha256"
)
manifest_path.write_text("\n".join(manifest_lines) + "\n", encoding="utf-8")

methodology = pd.DataFrame(
    [
        ["Sistema", f'{environment["system"]} {environment["release"]}'],
        ["Arquitectura / CPU", f'{environment["machine"]} · {environment["cpu"]}'],
        ["Python", environment["python"]],
        ["OpenSSL", environment["openssl_pkg_config"]],
        ["Hilos de la prueba principal", report["cracking"]["threads"]],
        ["Código C SHA-256", sha256_file(ROOT / "src" / "dni_sha256_enum.c")[:16] + "…"],
        ["Base sintética SHA-256", sha256_file(database)[:16] + "…"],
        ["Resultado SHA-256", sha256_file(report_path)[:16] + "…"],
    ],
    columns=["Elemento", "Valor"],
)
show_table(methodology)
print("Manifiesto generado con", len(manifest_lines), "entradas.")


## Capturas que sí aportan al artículo

| Figura | Sección del artículo | Pie sugerido |
| --- | --- | --- |
| `01_dominio_vs_hash.png` | La pregunta incómoda | «La salida tiene 256 bits, pero el dominio del DNI conserva solo 26,58 bits.» |
| `03_resultado_reconstruccion.png` | La prueba | «Enumeración completa sobre datos sintéticos: 100 millones de candidatos y 12/12 objetivos recuperados.» |
| `04_tres_caminos.png` | Los hallazgos | «Hash enumerable, clave de enlace y ofuscación reversible: tres caminos independientes hacia la atribución.» |
| `06_salt_global_conocida.png` | La solución | «Una sal global conocida cambia la tabla reutilizable, no el dominio que debe recorrerse.» |

`02_modelo_datos.png` es útil si el lector necesita comprender staging y
`link_key`. `05_perfiles_sinteticos.png` funciona como evidencia suplementaria,
pero no debe publicarse sin conservar su marca de datos sintéticos.

No captures las celdas de instalación ni rutas de Terminal. Para publicar,
inserta directamente los PNG de `results/figures_mac/`: tienen más resolución y
no exponen información local.
